# KAT-VAD v2.2 — **order diagnostics S1–S3** (no trunk training)

**Question.** Does *temporal order between steps* carry detection signal in the frozen features A3 feeds its trunk?
D6 already showed that order **inside** the 1.5 s VideoMAE window carries none. This notebook tests order **between**
steps at the same scale, before any training run is spent. Design: `core/docs/v2.2/KAT_VAD_PROPOSAL_v2.2.md` §3–§6
(review round 4). Plan and code choices O1–O15: `.project/plans/katvad-v2.2-order-diagnostics.md`.

| step | what | trains? | decides |
|---|---|---|---|
| 0 | setup, deps; refuses an upload without `order_diagnostics` or the v2.2 docs | — | — |
| 1 | preflight: splits, v2 data dir, CLIP + V2-S caches for every T2 source and CAP-dev clip; stage to VM disk | no | — |
| 2 | **S1** `m5` change score vs 0.5 and vs its order-free twin · Δu echo · clip-start SMD | no | TDP branch open? Δu admitted? H6 kept? |
| 3 | **S2** temporal probe, true order vs adjacency-free 5-step block shuffle, on `F0 = [x̃; ũ; p]` | probe only | stop, or continue |
| 4 | **S3** probe on `F0` / `F0+K` / `F0+K^π` — **only if S1a and S2 passed** (the tool refuses otherwise) | probe only | S5 eligible? |
| 5 | record → paste the three read-outs to Claude → advisor check | — | — |

**One bar for every gate** (proposal §6.1): cluster-bootstrap 95 % lower bound **> 0** and mean **≥ +0.01** on
DoTA-CAP-dev (protocol B). T2-val is reported, never decides. `f_2` Δ and per-seed Δs are printed beside every Δ.
DoTA-eval and DoTA-CAP-eval are never read. CPU is enough; a GPU only speeds up S2/S3.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/ (CLIP, VideoMAE)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # kat-vad/ (code), data/ (v2 dataset dir), outputs/
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V22_BAR_MIN'), 'this upload predates the v2.2 order diagnostics -- re-upload branch v2'
assert (REPO / 'core/tools/order_diagnostics.py').is_file(), 'upload has no core/tools/order_diagnostics.py'
PROPOSAL = REPO / 'core/docs/v2.2/KAT_VAD_PROPOSAL_v2.2.md'
assert 'S1a gates only the TDP branch' in PROPOSAL.read_text(encoding='utf-8'), \
    'upload has no round-4 v2.2 proposal'

DATASET = constants.DADA_ORIGIN_DATASET                    # 'DADA2000_orig'
ENC     = constants.VIDEOMAE_ENCODER_S                     # A3's video stream
T2      = constants.DATA_ROOT / DATASET
CENSUS  = T2 / 'counts' / 'resolved_census.json'
ORIG    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME
ANNOT   = ORIG / constants.DADA_ORIGIN_ANNOTATION_FILENAME
T2_CLIP = constants.CLIP_CACHE_DIR / DATASET               # s8, one file per source
VIDEO   = constants.VIDEO_CACHE_DIR
DOTA    = constants.DATA_ROOT / 'DoTA'
DOTA_S1 = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
V2_DATA = Path(DRIVE_V2) / 'data' / f'{DATASET}_v2'        # T2-train - T2-val windows
OUT     = constants.OUTPUT_ROOT / 'REPORTS' / 'v22_order'
LOCAL   = Path('/content/v22')
for p in (OUT, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
for name in ('T2_CLIP', 'VIDEO', 'DOTA_S1', 'V2_DATA', 'ANNOT', 'CENSUS', 'OUT'):
    print(f'{name:8s}', globals()[name])

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'no GPU -- fine, S2/S3 just run slower'
df -h /content | tail -1

In [ ]:
import json
import shutil
import subprocess
import time

import torch

ENV = dict(os.environ)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def run(cmd, log=None):
    # Stream a child's output here (and to ``log``); raise with its tail on failure.
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    # Copy a Drive dir (only ``ids`` + its manifests when given) to VM disk once, bit-identical (C2).
    t0 = time.time()
    if ids is None:
        if not dst.exists():
            shutil.copytree(src, dst)
    else:
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for m in src.glob('*.json'):
            shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst

## 1. Preflight + stage — every input exists before any probe minute is spent

In [ ]:
from core.data.v2_splits import load_split
from core.tools import dota_cap, e2d_probe, extract_video_features, order_diagnostics, v2_dataset

cap_dev = load_split(constants.V2_SPLIT_DOTA_CAP_DEV)          # sha1-checked; eval never loaded
assert len(cap_dev) == 569, len(cap_dev)
val_src = load_split(constants.V2_SPLIT_T2_VAL)
assert (V2_DATA / v2_dataset.MANIFEST_FILENAME).is_file(), f'no v2 dataset dir at {V2_DATA}'
meta = json.loads((V2_DATA / constants.META_FILENAME).read_text())
train_windows = json.loads((V2_DATA / constants.LABELS_TRAIN_FILENAME).read_text())
train_src = sorted({meta[w]['source'] for w in train_windows})
assert not set(train_src) & set(val_src), 'T2-val sources among the training windows'
census = json.loads(CENSUS.read_text())
assert all(v in census for v in train_src + val_src), 'a T2 source has no census entry'
assert ANNOT.is_file(), f'missing {ANNOT}'

S_T2, S_DOTA = e2d_probe.t2_video_dir(VIDEO, ENC), e2d_probe.dota_video_dir(VIDEO, ENC)
alignment = e2d_probe.dota_cap_alignment_sha256(constants.V2_SPLITS_DIR)
e2d_probe.check_manifest(S_DOTA, dota_cap.video_manifest(ENC, alignment))
e2d_probe.check_manifest(S_T2, extract_video_features.build_manifest(
    ENC, constants.FRAME_STRIDE, constants.VIDEOMAE_WEIGHTS_SHA256[ENC]))
t2_ids = train_src + list(val_src)
for d, ids, what in ((T2_CLIP, t2_ids, 'CLIP T2'), (S_T2, t2_ids, 'V2-S T2'),
                     (DOTA_S1, cap_dev, 'CLIP DoTA_s1_ncc'), (S_DOTA, cap_dev, 'V2-S DoTA-CAP')):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{what}: {len(miss)} missing, e.g. {miss[:3]}'
print(f'preflight OK | train sources {len(train_src)} ({len(train_windows)} windows) | '
      f'T2-val sources {len(val_src)} | DoTA-CAP-dev {len(cap_dev)}')

st = LOCAL / 'stage'
L_T2_CLIP = stage(T2_CLIP, st / 'clip_t2', t2_ids)
L_VIDEO = st / 'video'
stage(S_T2, L_VIDEO / ENC / S_T2.name, t2_ids)
stage(S_DOTA, L_VIDEO / ENC / S_DOTA.name, cap_dev)
L_DOTA_S1 = stage(DOTA_S1, st / 'DoTA_s1_ncc', cap_dev)
L_V2 = stage(V2_DATA, st / 'corpus_v2')

COMMON = ['--data-dir', L_V2, '--annotation', ANNOT, '--census', CENSUS,
          '--t2-clip-dir', L_T2_CLIP, '--video-root', L_VIDEO, '--dota-s1-dir', L_DOTA_S1,
          '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
          '--device', DEVICE, '--out-dir', OUT]
print('device', DEVICE)

## 2. S1 — free diagnostics (parameter-free, minutes)

- **S1a** `m5` = the 1.5 s causal max of the rate-normalised CLIP change. Both AUC − 0.5 and AUC − AUC(twin) must pass
  the bar, or the **TDP branch** (S3, S5) closes. **S2 runs either way.**
- **S1b** admits Δu to κ only if the exit-echo CI lies inside ±0.1 SD (equivalence). Otherwise κ is CLIP only.
- **S1c** withdraws the early-onset prediction if any of 5 summary channels has |SMD| > 0.1. It never stops the plan.

In [ ]:
t0 = time.time()
run([sys.executable, '-m', 'core.tools.order_diagnostics', 's1', *COMMON], log=OUT / 's1_run.log')
print(f'S1 in {(time.time() - t0) / 60:.1f} min\n')
print((OUT / order_diagnostics.readout_md('s1')).read_text())

## 3. S2 — D6-step probe (true order vs adjacency-free block shuffle)

20 probe fits (2 feature sets × 2 orders × 5 seeds). The gate reads `F0 = [x̃; ũ; p]`; the CLIP-only `[x̃; p]` read is
printed beside it, because `ũ_t` carries the past 1.5 s as a bag even after the shuffle. **If S2 fails, stop here:**
§9 row 1 applies, and nothing is trained.

In [ ]:
t0 = time.time()
run([sys.executable, '-m', 'core.tools.order_diagnostics', 's2', *COMMON], log=OUT / 's2_run.log')
print(f'S2 in {(time.time() - t0) / 60:.1f} min\n')
print((OUT / order_diagnostics.readout_md('s2')).read_text())

## 4. S3 — κ-probe (only if S1a **and** S2 passed)

`F0` vs `F0+K` (VALUE: explicit velocity/acceleration beyond what the probe learns itself) and `F0+K` vs `F0+K^π`
(ORDER: beyond each step's atypicality). Both must pass for S5 to be eligible. The cell skips itself otherwise, and
the tool also refuses to run.

In [ ]:
s1 = json.loads((OUT / order_diagnostics.readout_json('s1')).read_text())
s2 = json.loads((OUT / order_diagnostics.readout_json('s2')).read_text())
ok = constants.V22_GATE_OK
if s1['s1a']['verdict'] == ok and s2['verdict'] == ok:
    t0 = time.time()
    run([sys.executable, '-m', 'core.tools.order_diagnostics', 's3', *COMMON], log=OUT / 's3_run.log')
    print(f'S3 in {(time.time() - t0) / 60:.1f} min\n')
    print((OUT / order_diagnostics.readout_md('s3')).read_text())
else:
    print(f"S3 NOT RUN (pre-registered): S1a {s1['s1a']['verdict']}, S2 {s2['verdict']}")

## 5. Record — then **stop**

Paste `Thesis-V2/outputs/REPORTS/v22_order/s{1,2,3}_readout.md` back to Claude. Nothing below is decided here.

| Outcome | Thesis wording (proposal §9) | Next |
|---|---|---|
| **S2 fails** | "Neither frame order inside the 1.5 s window (D6) nor step order at the same scale (D6-step) carries detection signal … beyond VideoMAE's 1.5 s bag." | close; analysis chapter; 0 training runs |
| S2 passes, **S1a fails** | step order carries signal, but the size of change alone does not rank anomalous steps beyond their atypicality | TDP not built; S4 only with the advisor's approval |
| S2 passes, S3 fails (VALUE or ORDER) | explicit change features add nothing beyond the temporal probe, or only through atypicality | TDP not built; S4 only with the advisor's approval |
| S1a, S2, S3 all pass | S5 eligible | **advisor check** (proposal §10) before S4 (10 runs) and S5 (10 runs) |